In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

path = '/kaggle/input/datasets/adityakadiwal/water-potability/water_potability.csv'
df = pd.read_csv(path)

for col in ['ph', 'Sulfate', 'Trihalomethanes']:
    df[col] = df[col].fillna(df[col].median())

X = df.drop('Potability', axis=1)
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)
print("model trained")

model trained


In [2]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

@app.get("/ping")
def ping():
    return {"status": "ok"}

class WaterSample(BaseModel):
    ph: float
    Hardness: float
    Solids: float
    Chloramines: float
    Sulfate: float
    Conductivity: float
    Organic_carbon: float
    Trihalomethanes: float
    Turbidity: float

@app.post("/predict")
def predict(sample: WaterSample):
    data = np.array([[sample.ph, sample.Hardness, sample.Solids,
                      sample.Chloramines, sample.Sulfate, sample.Conductivity,
                      sample.Organic_carbon, sample.Trihalomethanes, sample.Turbidity]])
    prediction = int(model.predict(data)[0])
    return {"potable": bool(prediction), "prediction": prediction}

In [3]:
client = TestClient(app)

print(client.get("/ping").status_code, client.get("/ping").json())

sample = {
    "ph": 7.0, "Hardness": 200.0, "Solids": 20000.0,
    "Chloramines": 7.0, "Sulfate": 333.0, "Conductivity": 400.0,
    "Organic_carbon": 14.0, "Trihalomethanes": 66.0, "Turbidity": 4.0
}
r = client.post("/predict", json=sample)
print(r.status_code, r.json())

200 {'status': 'ok'}
200 {'potable': False, 'prediction': 0}
